# Clonamos el repositorio con los modelos y herramientas

In [1]:
!git clone https://github.com/dannasalazar11/Msc_thesis.git

Cloning into 'Msc_thesis'...
remote: Enumerating objects: 492, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 492 (delta 39), reused 0 (delta 0), pack-reused 427 (from 1)
Receiving objects: 100% (492/492), 50.49 MiB | 38.24 MiB/s, done.
Resolving deltas: 100% (316/316), done.


In [2]:
import sys
sys.path.append('/kaggle/working/Msc_thesis')

from gmrrnet_adhd.utils import get_segmented_data
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy('mixed_float16')

import tensorflow as tf
import numpy as np
import random
import os

# Establecer semilla
seed = 42

# Semillas para módulos principales
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)

2025-11-21 01:40:46.556625: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763689246.820820      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763689246.892842      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
import numpy as np
import random
from collections import defaultdict
from copy import deepcopy

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    cohen_kappa_score,
    roc_auc_score
)


def train_L24O_cv(model_builder, X, y, sbjs, model_args, compile_args, folds, model_name=''):
    all_fold_metrics = []
    models = {}

    for fold, (train_subjects, test_subjects) in enumerate(folds):
        print("-" * 50)
        print(f"Fold {fold+1}/{len(folds)}. Test subjects: {test_subjects}")
        print("-" * 50)

        train_idx = [i for i, sbj in enumerate(sbjs) if sbj in train_subjects]
        test_idx = [i for i, sbj in enumerate(sbjs) if sbj in test_subjects]

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        sbjs_test = [sbjs[i] for i in test_idx]

        # --- Build and Compile Model for each fold ---
        tf.keras.backend.clear_session() #<-- Clear session to prevent any state leakage
        
        # Re-set seeds for each fold for perfect reproducibility of weight initialization
        np.random.seed(seed + fold)
        random.seed(seed + fold)
        tf.random.set_seed(seed + fold)

        model = model_builder(**model_args)
        # Use a deepcopy to prevent the optimizer state from carrying over
        compile_args_local = deepcopy(compile_args)
        if callable(compile_args_local["optimizer"]):
            compile_args_local["optimizer"] = compile_args_local["optimizer"]()  # <-- aquí se reinicia
        model.compile(**compile_args_local)

        
        # --- Callbacks ---
        # EarlyStopping with restore_best_weights is crucial
        early_stopping = EarlyStopping(
            monitor='val_loss', patience=25, min_delta=1e-4, restore_best_weights=True, verbose=1
        )
        # ReduceLROnPlateau helps to fine-tune when learning stalls
        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1
        )

        # --- Train the Model ---
        model.fit(
            X_train, y_train,
            epochs=100,  #<-- Increased epochs to give LR scheduler more time to work
            validation_data=(X_test, y_test),
            verbose=0, #<-- Verbose=2 gives one line per epoch, cleaner log
            batch_size=16,
            # callbacks=[early_stopping, 
            #            reduce_lr]
        )

        # --- Predictions and Evaluation ---
        y_pred_probs = model.predict(X_test)
        print(y_pred_probs.shape)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = np.argmax(y_test, axis=1)

        # Overall fold metrics
        fold_metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'kappa': cohen_kappa_score(y_true, y_pred),
            'auc': roc_auc_score(y_true, y_pred_probs[:, 1]) # Use probabilities for AUC
        }
        print(f"\nFold {fold+1} Metrics: {fold_metrics}")
        all_fold_metrics.append(fold_metrics)
        models[fold] = model

        # Accuracy por sujeto de test
        subject_correct = defaultdict(list)
        for yt, yp, sbj in zip(y_true, y_pred, sbjs_test):
            subject_correct[sbj].append(int(yt == yp))

        subject_accuracies = {
            sbj: np.mean(subject_correct[sbj]) for sbj in subject_correct
        }

        print("Average accuracy per test subject:")
        for sbj in test_subjects:
            acc_sbj = subject_accuracies.get(sbj, None)
            if acc_sbj is not None:
                print(f"  {sbj}: {acc_sbj:.4f}")
                
        
    # --- Final Comprehensive Report ---
    print("\n" + "="*50)
    print("Cross-Validation Final Results")
    print("="*50)
    
    # Calculate mean and std dev for each metric
    mean_metrics = {}
    for key in all_fold_metrics[0].keys():
        values = [f[key] for f in all_fold_metrics]
        mean_metrics[f'mean_{key}'] = np.mean(values)
        mean_metrics[f'std_{key}'] = np.std(values)

    print("Individual Fold Accuracies:")
    for i, f in enumerate(all_fold_metrics):
        print(f"  Fold {i+1}: {f['accuracy']:.4f}")
        
    print("\nAverage Performance across all folds:")
    for key, value in mean_metrics.items():
        print(f"  {key}: {value:.4f}")
        
    return all_fold_metrics

# Importar base de datos segmentada (Segmentos de 4 seg con translape del 50%, es decir, de 2 seg)

In [4]:
X, y, sbjs = get_segmented_data()
X.shape, y.shape, len(sbjs)

((8213, 19, 512), (8213, 2), 8213)

# Importamos el modelo y definimos los hiperparámetros

In [5]:
from tensorflow.keras.losses import CategoricalCrossentropy, MeanSquaredError
from tensorflow.keras.optimizers import Adam
from gmrrnet_adhd.models.EEGNet import EEGNet

model_name = 'EEGNet'
model_args = {
    'Chans' : 19,
    'Samples' : 512,
    'nb_classes': 2,
    'dropoutRate': 0.5,
    'kernLength': 32,
    'F1': 8,
    'D': 2,
    'F2': 16,
    'norm_rate': 0.25,
    'dropoutType': 'Dropout'
}

compile_args = {
    'loss': CategoricalCrossentropy(),
    'optimizer': lambda: Adam(1e-2),  # función que retorna un nuevo optimizador
    'metrics': ['categorical_accuracy']
}


model = EEGNet(**model_args)

model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1763689263.794340      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1763689263.795031      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ keras_tensor_2CLONE (InputLayer)     │ (None, 19, 512, 1)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Conv2D_1 (Conv2D)                    │ (None, 19, 512, 8)          │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 19, 512, 8)          │              32 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Depth_wise_Conv2D_1                  │ (None, 1, 512, 16)          │             304 │
│ (DepthwiseConv2D)                    │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 1, 512, 16)          │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation (Activation)              │ (None, 1, 512, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ average_pooling2d (AveragePooling2D) │ (None, 1, 128, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1, 128, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Separable_Conv2D_1 (SeparableConv2D) │ (None, 1, 128, 16)          │             512 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 1, 128, 16)          │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_1 (Activation)            │ (None, 1, 128, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ average_pooling2d_1                  │ (None, 1, 16, 16)           │               0 │
│ (AveragePooling2D)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 1, 16, 16)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ output (Dense)                       │ (None, 2)                   │             514 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ out_activation (Activation)          │ (None, 2)                   │               0 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,746 (6.82 KB)

 Trainable params: 1,666 (6.51 KB)

 Non-trainable params: 80 (320.00 B)

# Resultados - Leave 24 Subjects Out

In [6]:
import os

import pickle

with open("/kaggle/input/ieee-tdah-control-database/folds.pkl", "rb") as f:
    folds = pickle.load(f)

In [7]:
import numpy as np

results = {}

for i in range(10):
    result = train_L24O_cv(EEGNet, X, y, sbjs, model_args, compile_args, folds)
    results[i] = result

--------------------------------------------------
Fold 1/5. Test subjects: ['v28p', 'v274', 'v1p', 'v231', 'v22p', 'v29p', 'v206', 'v238', 'v31p', 'v35p', 'v177', 'v200', 'v112', 'v113', 'v48p', 'v140', 'v131', 'v125', 'v55p', 'v143', 'v43p', 'v305', 'v134', 'v114']
--------------------------------------------------


I0000 00:00:1763689269.541636      70 service.cc:148] XLA service 0x7b18c400e7c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1763689269.542451      70 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1763689269.542473      70 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1763689269.913077      70 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1763689273.846216      70 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
(1457, 2)

Fold 1 Metrics: {'accuracy': 0.7803706245710363, 'recall': 0.7896889679339043, 'precision': 0.8415735014536889, 'kappa': 0.5682891011146483, 'auc': 0.8506313531801772}
Average accuracy per test subject:
  v28p: 0.0000
  v274: 0.9848
  v1p: 1.0000
  v231: 0.0263
  v22p: 1.0000
  v29p: 0.9785
  v206: 0.0000
  v238: 1.0000
  v31p: 1.0000
  v35p: 1.0000
  v177: 0.1250
  v200: 0.9375
  v112: 1.0000
  v113: 1.0000
  v48p: 1.0000
  v140: 1.0000
  v131: 1.0000
  v125: 1.0000
  v55p: 0.9815
  v143: 1.0000
  v43p: 1.0000
  v305: 1.0000
  v134: 1.0000
  v114: 1.0000
--------------------------------------------------
Fold 2/5. Test subjects: ['v18p', 'v39p', 'v234', 'v32p', 'v190', 'v6p', 'v254', 'v204', 'v24p', 'v183', 'v246', 'v219', 'v298', 'v41p', 'v47p', 'v308', 'v52p', 'v300', 'v59p', 'v299', 'v302', 'v51p', 'v109', 'v127']
--------------------------------------------------


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
(1668, 2)

Fold 2 Metrics: {'accuracy': 0.6492805755395683, 'recall': 0.6172576721897786, 'precision': 0.6736840287372202, 'kappa': 0.24801403201593086, 'auc': 0.7071378290657834}
Average accuracy per test subject:
  v18p: 1.0000
  v39p: 1.0000
  v234: 1.0000
  v32p: 1.0000
  v190: 1.0000
  v6p: 0.6866
  v254: 1.0000
  v204: 0.0513
  v24p: 1.0000
  v183: 0.9718
  v246: 0.9634
  v219: 1.0000
  v298: 0.0000
  v41p: 0.9149
  v47p: 0.9512
  v308: 0.0000
  v52p: 1.0000
  v300: 1.0000
  v59p: 0.0000
  v299: 0.0000
  v302: 0.0000
  v51p: 0.0333
  v109: 0.2459
  v127: 0.0179
--------------------------------------------------
Fold 3/5. Test subjects: ['v215', 'v3p', 'v209', 'v37p', 'v213', 'v15p', 'v284', 'v181', 'v19p', 'v34p', 'v263', 'v244', 'v138', 'v121', 'v46p', 'v54p', 'v120', 'v310', 'v147', 'v50p', 'v56p', 'v107', 'v297', 'v108']
--------------------------------------------------


...

56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
(1768, 2)

Fold 3 Metrics: {'accuracy': 0.6470588235294118, 'recall': 0.7064910630291628, 'precision': 0.765237020316027, 'kappa': 0.3594135947630446, 'auc': 0.9599754475157289}
Average accuracy per test subject:
  v215: 0.9268
  v3p: 0.0000
  v209: 0.8547
  v37p: 1.0000
  v213: 1.0000
  v15p: 0.0060
  v284: 0.4746
  v181: 0.1250
  v19p: 0.0000
  v34p: 0.9867
  v263: 0.9859
  v244: 0.0199
  v138: 1.0000
  v121: 1.0000
  v46p: 1.0000
  v54p: 1.0000
  v120: 1.0000
  v310: 1.0000
  v147: 1.0000
  v50p: 1.0000
  v56p: 1.0000
  v107: 1.0000
  v297: 1.0000
  v108: 1.0000
--------------------------------------------------
Fold 4/5. Test subjects: ['v227', 'v8p', 'v236', 'v14p', 'v196', 'v27p', 'v33p', 'v179', 'v173', 'v10p', 'v265', 'v20p', 'v57p', 'v45p', 'v111', 'v115', 'v53p', 'v118', 'v123', 'v44p', 'v149', 'v303', 'v116', 'v151']
--------------------------------------------------


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
(1687, 2)

Fold 4 Metrics: {'accuracy': 0.8363959691760522, 'recall': 0.8275603403269021, 'precision': 0.8369316337670768, 'kappa': 0.6623866675174458, 'auc': 0.8804438248337664}
Average accuracy per test subject:
  v227: 0.9537
  v8p: 1.0000
  v236: 1.0000
  v14p: 0.0597
  v196: 1.0000
  v27p: 1.0000
  v33p: 1.0000
  v179: 1.0000
  v173: 1.0000
  v10p: 0.2963
  v265: 1.0000
  v20p: 1.0000
  v57p: 1.0000
  v45p: 0.8049
  v111: 1.0000
  v115: 1.0000
  v53p: 0.0000
  v118: 1.0000
  v123: 1.0000
  v44p: 0.0000
  v149: 0.2923
  v303: 1.0000
  v116: 1.0000
  v151: 1.0000
--------------------------------------------------
Fold 5/5. Test subjects: ['v279', 'v30p', 'v288', 'v286', 'v250', 'v12p', 'v38p', 'v25p', 'v21p', 'v40p', 'v198', 'v270', 'v117', 'v306', 'v309', 'v110', 'v42p', 'v58p', 'v307', 'v133', 'v304', 'v129', 'v49p', 'v60p']
--------------------------------------------------


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
(1633, 2)

Fold 5 Metrics: {'accuracy': 0.8873239436619719, 'recall': 0.8843090016661413, 'precision': 0.9094784569619039, 'kappa': 0.7731591717915074, 'auc': 0.9808573873103075}
Average accuracy per test subject:
  v279: 1.0000
  v30p: 1.0000
  v288: 1.0000
  v286: 1.0000
  v250: 1.0000
  v12p: 1.0000
  v38p: 0.9895
  v25p: 1.0000
  v21p: 1.0000
  v40p: 1.0000
  v198: 1.0000
  v270: 1.0000
  v117: 0.9495
  v306: 0.0143
  v309: 1.0000
  v110: 1.0000
  v42p: 1.0000
  v58p: 1.0000
  v307: 1.0000
  v133: 0.9825
  v304: 1.0000
  v129: 0.0000
  v49p: 0.7344
  v60p: 0.0000

Cross-Validation Final Results
Individual Fold Accuracies:
  Fold 1: 0.5903
  Fold 2: 0.7380
  Fold 3: 0.6471
  Fold 4: 0.8364
  Fold 5: 0.8873

Average Performance across all folds:
  mean_accuracy: 0.7398
  std_accuracy: 0.1114
  mean_recall: 0.7531
  std_recall: 0.0962
  mean_precision: 0.8034
  std_precision: 0.0625
  mean_kappa: 0.4954
  std_kappa: 0.2033
  mean_auc: 0.8866
  st

In [8]:
for i in range(10):
    result = results[i]
    accs = []
    for r in result:
        accs.append(r['accuracy'])
    
    print(i, '->', np.mean(accs))

0 -> 0.8134857559581053
1 -> 0.8033266769925174
2 -> 0.7275907874299179
3 -> 0.6938297897321715
4 -> 0.7591751713484816
5 -> 0.7896549723421222
6 -> 0.6846001242859687
7 -> 0.7465147898378053
8 -> 0.803165178012328
9 -> 0.7398084550317829


In [9]:
import pickle

with open(f'results_L24SO_{model_name}.pkl', 'wb') as f:
    pickle.dump(results, f)